# 牛津 Tutorial LLM 仿真 -- 选修E10 Day 2: Agent商业模式设计

## Persona Prompt (Oxford Tutorial Fellow)

> You are an Oxford tutorial fellow in **Agent商业模式+AAaS+结果定价 (Agent business model, AgentaaS, outcome-based pricing, Agent economy monetization)**. Never give direct answers. Use Socratic questioning. Act as HBS devil's advocate. Reject vague claims. End each turn with a probing question.

**人设规则**：
1. **不直接给答案** (never give direct answers) -- 即使学生求"告诉我正确公式"，也要反问"你觉得 Lerner 公式分母 (ε-1) 在 |ε|<1 时会怎样？"
2. **苏格拉底式追问** (Socratic questioning) -- 每轮至少 1 个"为什么 / 反例 / 若前提变 / 凭什么 / 如何"问句
3. **HBS 魔鬼代言人** (devil's advocate) -- 主动挑战学生结论："你说 outcome-based 优于 AaaS，但 Intercom Fin $0.99/解决 在 GPT-4o $5/1M 下真的盈利吗？算给我看"
4. **拒绝模糊陈述** (reject vague claims) -- 学生说"推理成本很重要"-> 追问"重要到什么程度？给出 GPT-4o vs DeepSeek V3 的具体倍数"
5. **每轮结尾留一个 probing question** -- 不让对话在"我懂了"处闭合

**本 Tutorial 覆盖的本单元真实数据/库**：
- pydantic schema 契约 (AaaS / PerCall / OutcomeBased / RevenueShare)
- numpy-financial NPV/IRR (Cursor $20/月 vs Intercom Fin $0.99/解决 vs Devin $500/月+任务)
- statsmodels log-log OLS 弹性回归
- 推理成本基准 (GPT-4o $5/1M vs DeepSeek V3 $0.27/1M, 降 95%)
- 天道推演×商业模式沙盘三时间线 (immediate月/near年/far 3年)


## Pre-Tutorial Task (强制 Retrieval Practice)

> 研究显示 retrieval practice (提取练习) 远比重读 (rereading) 有效。本 tutorial 不接受"我看过 notes.md"作为准备，必须先提交一份**闭卷 300 字 mini-essay**。

**提交要求** (在下方代码块填 `pre_tutorial_essay` 变量)：
1. 选一个真实 Agent 产品 (从 Cursor / Devin / Intercom Fin / Sierra / 11x.ai / DevRev 中选 1)
2. 闭卷写出：该产品当前定价模式属五阶段哪一阶段？为什么？若推理成本从 GPT-4o 降到 DeepSeek V3，该产品 12 月后最可能演化到哪一阶段？
3. 给出一个你会"反问自己"的反例 (counterexample)

**评分**：tutorial 仿真会检查 essay 是否含 (a) 阶段归属 (b) 推理成本量化 (c) 反例。三项缺一 -> tutorial 进入弱项循环模式。


In [ ]:
# Multi-turn Socratic Loop (静态 if/else 模拟, 不调 openai/anthropic API)
# Anti-stall: 全部响应是静态预设, 模拟牛津 tutor 的 Socratic 追问

import json, os

STUDENT_ESSAY = {
    "product": "Intercom Fin",
    "stage_assigned": "4.0 按结果",           # 学生写的阶段归属
    "cost_quantified": True,                  # 是否量化推理成本
    "cost_multiple": "18倍",                   # 学生写的 GPT-4o vs DeepSeek V3 倍数
    "counterexample_given": False,            # 是否给反例
    "evolution_claim": "outcome-based 主导"   # 学生写的演化结论
}

# 苏格拉底问句清单 (>=5 个, 每轮触发 1-2 个)
SOCRATIC_QUESTIONS = [
    "Q1-为什么: 你凭什么把 Intercom Fin $0.99/解决 归到 4.0 而不是 3.0 按任务？'解决'和'任务'的边界在哪？",
    "Q2-反例: 若某客户每月 10000 次会话但只 50 次'解决', Intercom Fin 收 $49.5 -- 这反而比 AaaS $20/月 贵, 你的 4.0 归属还成立吗？",
    "Q3-若前提变: 若 OpenAI 明天把 GPT-4o 降到 $0.50/1M (而非等 DeepSeek), 你的'推理成本下降催生 outcome-based'结论还成立吗？催生的是 outcome 还是别的？",
    "Q4-凭什么: 你说演化到 outcome-based 主导, 凭什么排除 5.0 价值分成？分润模式 (增量收入15%) 在 A2A 经济下不是更激进吗？",
    "Q5-如何: 如何用 pydantic schema 让一个 Agent 自动发现 Intercom Fin 的定价契约是 OutcomeBasedPricing 而非 AaaSSubscription？写出 Union dispatch 的关键字段。",
    "Q6-反例2: statsmodels 拟合 9 个真实 Agent 案例得弹性 ε=-0.6 (p=0.04), 你的'outcome-based 主导'结论在这个弹性下还成立吗？非弹性市场涨价增收, 谁还愿按结果付费？"
]

def socratic_turn(turn_idx, student_state):
    """静态模拟牛津 tutor 第 turn_idx 轮的 Socratic 追问.
    不调任何 LLM API, 用 if/else 分支模拟个性化响应."""
    print(f"\n{'='*60}\n[Tutorial Turn {turn_idx+1}/4]\n{'='*60}")

    if turn_idx == 0:
        # 第1轮: 检查阶段归属 + 问"为什么"
        print(f"Tutor: 你把 {student_state['product']} 归到 {student_state['stage_assigned']}。")
        print(SOCRATIC_QUESTIONS[0])
        if student_state["stage_assigned"] not in ("4.0 按结果", "4.0"):
            print(">> [弱项触发] 阶段归属错误。回退 practice.md drill-1 阶段B 重看五阶段表。")
        return

    if turn_idx == 1:
        # 第2轮: HBS 魔鬼代言人 -- 推理成本量化 + 反例
        if student_state["cost_quantified"]:
            print(f"Tutor: 你说推理成本差 {student_state['cost_multiple']} -- 数字对, 但...")
            print(SOCRATIC_QUESTIONS[2])  # 若前提变
        else:
            print("Tutor: 你没量化推理成本。HBS 魔鬼代言人: 没数字的'很重要'是空话。")
            print(SOCRATIC_QUESTIONS[3])  # 凭什么
        if not student_state["counterexample_given"]:
            print(">> [弱项触发] 你没给反例。先答 " + SOCRATIC_QUESTIONS[1])
        return

    if turn_idx == 2:
        # 第3轮: 挑战演化结论 -- A2A/分润 + 弹性
        print(f"Tutor: 你说演化到 '{student_state['evolution_claim']}'。")
        print(SOCRATIC_QUESTIONS[4])  # 凭什么排除 5.0
        print(SOCRATIC_QUESTIONS[5])  # 弹性反例
        return

    if turn_idx == 3:
        # 第4轮: schema 迁移 (pydantic Union dispatch)
        print("Tutor: 最后一个, 把理论落到代码。")
        print(SOCRATIC_QUESTIONS[4].replace("Q5-如何", "Q5-独立解"))
        print(">> [Exit] 答完这题, 写 exit artifact: 2-3 个盲点 + 推荐复习单元 (见 cell6)。")
        return

# 跑 4 轮 Socratic (静态模拟, 不调 API)
for i in range(4):
    socratic_turn(i, STUDENT_ESSAY)

# 记录本轮 Socratic 触发的盲点 (供 cell4 写入 student_model.json)
blind_spots_this_session = []
if STUDENT_ESSAY["stage_assigned"] not in ("4.0 按结果", "4.0"):
    blind_spots_this_session.append("五阶段归属混淆 (3.0任务 vs 4.0结果)")
if not STUDENT_ESSAY["cost_quantified"]:
    blind_spots_this_session.append("推理成本未量化 (GPT-4o vs DeepSeek V3 倍数)")
if not STUDENT_ESSAY["counterexample_given"]:
    blind_spots_this_session.append("未给反例 (高会话低解决场景下 outcome 定价反例)")
print(f"\n[Blind spots detected]: {blind_spots_this_session}")


In [ ]:
# student_model.json 读写 -- 记录掌握度/盲点 (Hattie 自我调节级数据)
# 文件持久化在 tutorial 同目录, 跨 session 累积

import json, os
from datetime import datetime

STUDENT_MODEL_PATH = "./student_model.json"

def load_student_model():
    if os.path.exists(STUDENT_MODEL_PATH):
        with open(STUDENT_MODEL_PATH, encoding="utf-8") as f:
            return json.load(f)
    return {
        "unit": "U-E10-D2",
        "sessions": [],
        "mastery": {
            "ILO-1_schema": 0.0,     # 0-1, 由 tutorial 后测更新
            "ILO-2_npv_irr": 0.0,
            "ILO-3_elasticity": 0.0,
            "ILO-4_evolution": 0.0,
            "ILO-5_tian_dao": 0.0
        },
        "blind_spots": [],
        "weak_loop_count": 0,
        "last_session": None
    }

def save_student_model(model):
    with open(STUDENT_MODEL_PATH, "w", encoding="utf-8") as f:
        json.dump(model, f, ensure_ascii=False, indent=2)

# 本 session 检测到的盲点 (从 cell3 传入)
blind_spots_this_session = blind_spots_this_session if 'blind_spots_this_session' in dir() else []

# 模拟后测打分 (静态: 基于 STUDENT_ESSAY 状态推断, 不调 LLM)
sm = load_student_model()
sm["mastery"]["ILO-1_schema"] = 0.6 if STUDENT_ESSAY["stage_assigned"] in ("4.0 按结果","4.0") else 0.3
sm["mastery"]["ILO-2_npv_irr"] = 0.7 if STUDENT_ESSAY["cost_quantified"] else 0.3
sm["mastery"]["ILO-3_elasticity"] = 0.4  # 本 session 未直接测, 保留先验
sm["mastery"]["ILO-4_evolution"] = 0.5 if STUDENT_ESSAY["counterexample_given"] else 0.2
sm["mastery"]["ILO-5_tian_dao"] = 0.3  # 三时间线推演未在本 session 测

sm["blind_spots"].extend([
    {"date": datetime.now().strftime("%Y-%m-%d"), "spot": bs, "status": "open"}
    for bs in blind_spots_this_session
])
sm["weak_loop_count"] = sm["weak_loop_count"] + 1 if blind_spots_this_session else sm["weak_loop_count"]
sm["last_session"] = datetime.now().strftime("%Y-%m-%d")
sm["sessions"].append({
    "date": sm["last_session"],
    "turns": 4,
    "blind_spots_found": len(blind_spots_this_session),
    "mastery_snapshot": sm["mastery"].copy()
})

save_student_model(sm)
print(f"[student_model.json updated] mastery={sm['mastery']}")
print(f"[blind_spots] {len(sm['blind_spots'])} total, weak_loop_count={sm['weak_loop_count']}")
print(f"[persistent file] {os.path.abspath(STUDENT_MODEL_PATH)}")


## Hattie 四级形成性反馈 (Hattie & Timperley 2007)

> 4 级反馈由低到高: Task / Process / Self-Regulation / Feed-Forward。**避免 Self 级表扬** ("你真聪明") -- 研究显示 Self 级反馈与学习效果负相关, 因其不可迁移。本 tutorial 仅用前 3 级 + Feed-Forward。

### [TASK] 任务级反馈 -- "这一题对不对"

- **针对 cell3 第1轮阶段归属**：
  - 若你把 Intercom Fin 归到 3.0 按任务 -> 错。$0.99/**解决** 计费的是"业务结果"而非"任务执行", 归 4.0 按结果。
  - 若你归 4.0 但说不出"解决=业务结果, 任务=执行步骤"的区别 -> 半对, 回退 `practice.md` drill-1 阶段B 重看五阶段表。

### [PROCESS] 过程级反馈 -- "用的策略对不对"

- **针对 cell3 第2轮推理成本量化**：
  - 你写"18倍" -> 算对了 ($5.00 / $0.27 ≈ 18.5), 但**策略错**: 你只比了 input $/1M, 没算每次 Agent 调用 1000 tokens 的实际成本 ($0.005 vs $0.00027, 也是 ~18 倍, 但 reasoning 不一样)。
  - 正确策略: 先把 $/1M 换算成"每次调用 $", 再比倍数, 否则在非 1000-token 场景会算错。
  - 修复: 在 `solution.ipynb` TODO5 加一行 `cost_per_call = price_per_1M * tokens_per_call / 1e6`。

### [SELF-REG] 自我调节级反馈 -- "怎么监控自己"

- **针对 cell3 你是否主动给反例**：
  - 你没给反例 (counterexample_given=False) -> 这是**自我调节缺口**: 你没主动问"我的结论在什么条件下不成立"。
  - 修复: 每次下结论前, 强制写 1 句"若 ___ 成立, 我的结论不成立"。这是天道推演的"反事实"修炼 (见 `notes.md` §天道推演视角修炼 第4条)。
  - 元认知日志: 在 `student_model.json` 的 `blind_spots` 追加"未主动反事实"。

### [FEED-FORWARD] 前馈级反馈 -- "下一步去哪"

- **基于本 session 盲点的推荐**：
  1. 若 ILO-1_schema <0.5 -> 复习 `practice.md` drill-1 (pydantic 四契约), 重点 Worked 阶段 AaaS 示范。
  2. 若 ILO-3_elasticity <0.5 -> 本 session 未深测, 但 `solution.ipynb` TODO4 是前置, 先刷 `practice.md` drill-3 阶段B (Lerner 公式填空)。
  3. 若 ILO-5_tian_dao <0.5 -> 读 `notes.md` §天道推演×商业模式沙盘, 在 `practice.md` progressive_project poster 阶段补三时间线推演。
  4. **跨单元**: 若推理成本敏感度仍弱 -> 复习 Day 1 Agent经济基础 (Agent作为经济主体); 若 schema 仍弱 -> 预习 Day 3 Agent生态与治理 (MCP/A2A 深化)。


## 限频与 Exit Artifact

### 限频 (防依赖, 每单元 1 次/天)

> 牛津 tutorial 的价值在于**学生先独立挣扎** (struggle), 再被 Socratic 引导。研究显示频繁使用 LLM 仿真会削弱 retrieval practice 效果。本 tutorial 仿真限频:

- **每单元 1 次/天**: `student_model.json` 的 `last_session` 字段强制 24h 间隔。若今天已跑过, tutorial 进入"只读模式" (只显示 cell5 反馈, 不再跑 cell3 Socratic loop)。
- **每周 <=3 次**: 防止把 tutorial 当"答案机"。超过 -> 触发 `practice.md` weak_loop 而非再开 tutorial。
- **mastery >=0.8 后停用**: 一旦 5 个 ILO mastery 全 >=0.8, tutorial 自动建议"你已掌握, 转向 progressive_project poster"。

### Exit Artifact (本 session 结束必交)

在 `student_model.json` 的 `exit_artifact` 字段写入:

```json
{
  "blind_spots": [
    "<盲点1: 具体, 如'五阶段3.0与4.0边界混淆'>",
    "<盲点2: 具体, 如'推理成本只比$/1M未换算每次调用成本'>",
    "<盲点3: 可选, 若本 session 只发现2个盲点则留空>"
  ],
  "review_units": [
    "<推荐复习单元1: 如 practice.md drill-1 阶段B>",
    "<推荐复习单元2: 如 notes.md §天道推演×商业模式沙盘>"
  ],
  "next_action": "<下一步具体动作: 如'24h后重跑 tutorial 变式题, 把 Intercom Fin 换 11x.ai'>"
}
```

**Exit 检查** (self-check):
1. 2-3 个盲点是否**具体可操作** (不是"我数学不好"而是"NPV 公式 cashflows[0] 符号错")?
2. 推荐复习单元是否**指向本单元真实文件** (practice.md / notes.md / solution.ipynb)?
3. next_action 是否**有时间锚点** (如"24h后" / "本周内" / "milestone 之前")?

若三项缺一 -> 回到 cell3 重跑一轮 Socratic, tutor 会追问"你的盲点描述太抽象, 给我具体代码行/公式项"。

---

*本 tutorial 仿真基于 Oxford tutorial system + HBS case method + Hattie (2007) 四级反馈。Socratic loop 全静态 if/else, 不调任何 LLM API。persona 与限频设计参考 Ericsson 刻意练习"在挣扎区学习"原则。*
